In [6]:
long_document = """
University Academic Handbook

Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.

Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.

Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.

Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.

Examination Rules:
Students must carry their hall ticket to the exam hall.
Electronic devices are not allowed during exams.
"""

In [7]:
evaluation_data = [
    {
        "query": "What attendance is required for semester exams?",
        "expected_keyword": "75% attendance"
    },
    {
        "query": "When do hostel gates close?",
        "expected_keyword": "9 PM on weekdays"
    },
    {
        "query": "What is the refund deadline for fees?",
        "expected_keyword": "15 days after the start of classes"
    },
    {
        "query": "How many books can students borrow?",
        "expected_keyword": "4 books"
    },
    {
        "query": "Are electronic devices allowed in exams?",
        "expected_keyword": "Electronic devices are not allowed"
    }
]

In [8]:
def section_aware_chunking(text):
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    return chunks

In [9]:
section_chunks = section_aware_chunking(long_document)

# Install Chroma

In [1]:
pip install chromadb

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached orjson-3.11.7-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
   ---------------------------------------- 0.0/21.9 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/21.9 MB 15.2 MB/s eta 0:00:02
   ----------- ---------------------------- 6.6/21.9 MB 15.8 MB/s eta 0:00:01
   ----------------------- ---------------- 13.1/21.9 MB 21.2 MB/s eta 0:00:01
   ------------------------------------ --- 19.9/21.9 MB 23.8 MB/s eta 0:00:01
   ---------------------------------------- 21.9/21.9 MB 21.1 MB/s eta 0:00:00
Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl (150 kB)
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   -------------------------------------- - 4.7/4.9 MB 33.2 MB/s eta 0:00:01
   ---------------------------------------- 4.9/4.9 MB 23.7 MB/


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Keep one setup fixed

To make comparison fair, use:

1) section-aware chunks
2) MiniLM embeddings

So only the vector database changes:

1) FAISS
2) Chroma

# Import libraries

In [2]:
import chromadb
import time

# Build a Chroma collection

In [3]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_section_chunks")

# Generate MiniLM embeddings for the chunks

In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
# Embedding helper
def get_embeddings(texts, model):
    embeddings = model.encode(texts)
    return np.array(embeddings, dtype="float32")

minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
minilm_chunk_embeddings = get_embeddings(section_chunks, minilm_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9828.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Add chunks to Chroma

Chroma stores:

1) ids
2) documents
3) embeddings

In [14]:
ids = [f"chunk_{i}" for i in range(len(section_chunks))]

collection.add(
    ids=ids,
    documents=section_chunks,
    embeddings=minilm_chunk_embeddings.tolist()
)

# Query Chroma

In [16]:
# Query embedding
def embed_query(query, model):
    return np.array(model.encode([query]), dtype="float32")

In [17]:
query = "When do hostel gates close?"
query_embedding = embed_query(query, minilm_model)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=2
)

print(results["documents"])

[['Hostel Rules:\nHostel gates close at 9 PM on weekdays and 10 PM on weekends.\nStudents must carry their ID cards while entering the hostel.', 'Library Policy:\nStudents can borrow up to 4 books at a time.\nBooks must be returned within 14 days to avoid a fine.']]


# Make a retrieval helper for Chroma

In [18]:
def retrieve_top_k_chroma(query, collection, model, k=1):
    query_embedding = embed_query(query, model)
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )
    
    retrieved_docs = results["documents"][0]
    distances = results["distances"][0]
    
    output = []
    for rank, (doc, dist) in enumerate(zip(retrieved_docs, distances), start=1):
        output.append({
            "rank": rank,
            "chunk": doc,
            "distance": dist
        })
    
    return output

# Evaluate Chroma Hit@k

In [19]:
def evaluate_hit_at_k_chroma(evaluation_data, collection, model, k=1):
    hits = 0
    
    for item in evaluation_data:
        query = item["query"]
        expected_keyword = item["expected_keyword"].lower()
        
        results = retrieve_top_k_chroma(query, collection, model, k=k)
        
        found = any(expected_keyword in result["chunk"].lower() for result in results)
        if found:
            hits += 1
    
    return hits / len(evaluation_data)

# Benchmark indexing time

FAISS indexing time

In [23]:
import faiss
# FAISS builder
def build_faiss_index_from_embeddings(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [24]:
start = time.time()
faiss_embeddings = get_embeddings(section_chunks, minilm_model)
faiss_index = build_faiss_index_from_embeddings(faiss_embeddings)
faiss_index_time = time.time() - start

print("FAISS indexing time:", round(faiss_index_time, 6), "seconds")

FAISS indexing time: 0.050677 seconds


Chroma indexing time

In [25]:
start = time.time()

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="rag_section_chunks_benchmark")

ids = [f"chunk_{i}" for i in range(len(section_chunks))]
chroma_embeddings = get_embeddings(section_chunks, minilm_model)

collection.add(
    ids=ids,
    documents=section_chunks,
    embeddings=chroma_embeddings.tolist()
)

chroma_index_time = time.time() - start

print("Chroma indexing time:", round(chroma_index_time, 6), "seconds")

Chroma indexing time: 0.060819 seconds


# Benchmark query latency

In [26]:
# Retrieval helper
def retrieve_top_k_with_query_embedding(query_embedding, chunks, index, k=1):
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for rank, i in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk": chunks[i],
            "distance": distances[0][rank - 1]
        })
    return results

FAISS query latency

In [27]:
query = "What is the refund deadline for fees?"

start = time.time()
query_embedding = embed_query(query, minilm_model)
results = retrieve_top_k_with_query_embedding(query_embedding, section_chunks, faiss_index, k=2)
faiss_query_time = time.time() - start

print("FAISS query time:", round(faiss_query_time, 6), "seconds")
print(results)

FAISS query time: 0.022407 seconds
[{'rank': 1, 'chunk': 'Fee Refund Policy:\nThe refund deadline for semester fees is 15 days after the start of classes.\nNo refund will be issued after this deadline except under extraordinary circumstances.', 'distance': np.float32(0.5719112)}, {'rank': 2, 'chunk': 'Library Policy:\nStudents can borrow up to 4 books at a time.\nBooks must be returned within 14 days to avoid a fine.', 'distance': np.float32(1.327609)}]


Chroma query latency

In [28]:
start = time.time()
results = retrieve_top_k_chroma(query, collection, minilm_model, k=2)
chroma_query_time = time.time() - start

print("Chroma query time:", round(chroma_query_time, 6), "seconds")
print(results)

Chroma query time: 0.024798 seconds
[{'rank': 1, 'chunk': 'Fee Refund Policy:\nThe refund deadline for semester fees is 15 days after the start of classes.\nNo refund will be issued after this deadline except under extraordinary circumstances.', 'distance': 0.571911096572876}, {'rank': 2, 'chunk': 'Library Policy:\nStudents can borrow up to 4 books at a time.\nBooks must be returned within 14 days to avoid a fine.', 'distance': 1.3276088237762451}]


# Compare Hit@1 / Hit@2

In [29]:
# Evaluation
def evaluate_hit_at_k_local(evaluation_data, chunks, index, model, k=1):
    hits = 0
    
    for item in evaluation_data:
        query = item["query"]
        expected_keyword = item["expected_keyword"].lower()
        
        query_embedding = embed_query(query, model)
        results = retrieve_top_k_with_query_embedding(query_embedding, chunks, index, k=k)
        
        found = any(expected_keyword in result["chunk"].lower() for result in results)
        if found:
            hits += 1
    
    return hits / len(evaluation_data)

In [34]:
faiss_hit1 = evaluate_hit_at_k_local(evaluation_data, section_chunks, faiss_index, minilm_model, k=1)
faiss_hit2 = evaluate_hit_at_k_local(evaluation_data, section_chunks, faiss_index, minilm_model, k=2)

chroma_hit1 = evaluate_hit_at_k_chroma(evaluation_data, collection, minilm_model, k=1)
chroma_hit2 = evaluate_hit_at_k_chroma(evaluation_data, collection, minilm_model, k=2)

print("FAISS Hit@1:", round(faiss_hit1, 2))
print("FAISS Hit@2:", round(faiss_hit2, 2))
print("Chroma Hit@1:", round(chroma_hit1, 2))
print("Chroma Hit@2:", round(chroma_hit2, 2))

FAISS Hit@1: 1.0
FAISS Hit@2: 1.0
Chroma Hit@1: 1.0
Chroma Hit@2: 1.0
